# STATE on merfish, size=100, quality=1.0 — verbose training

Single-config smoke-test for the new training settings:
- val every 500 optimizer steps (clamped to batches/epoch on tiny data)
- early stopping on `validation/val_loss`, patience=5
- fixed step budget of 15,000 optimizer steps (no max_epochs cap); Lightning's epoch count is derived from max_steps so it stops when the step budget is hit

Subprocess stdout/stderr stream live into the cell output so the full STATE
Lightning training log is visible.

Naming note: filename uses the `plotting_` prefix per the repo convention
for `.ipynb`, even though this notebook is interactive compute.


In [1]:
from scaling_laws.prepare.data import Experiments, ExperimentJobIterator

PATH_TO_DATA_DIR = "/home/igor/noise_scaling/data"
DATASET = "merfish"
SIZE = 100
QUALITY = 1.0
SIGNAL_COLUMNS = ["cur_idx", "ng_idx"]
SEED = 1404
DEVICE = 0


/home/igor/miniconda3/envs/modeling/lib/python3.10/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


/home/igor/miniconda3/envs/modeling/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
experiments = Experiments(
    datasets=[DATASET],
    sizes=[SIZE],
    qualities=[QUALITY],
    algos=["State"],
    path_to_data_dir=PATH_TO_DATA_DIR,
    signal_columns=SIGNAL_COLUMNS,
    device=DEVICE,
    seed=SEED,
)

# Reproduce the (max_epochs, early_stopping_patience, max_steps) the
# orchestrator would have computed for this (dataset, size). The per-size
# config logic lives on ExperimentJobIterator, not Experiments, so build a
# one-shot iterator just to call it.
_iter = ExperimentJobIterator(
    datasets=[DATASET],
    sizes=[SIZE],
    qualities=[QUALITY],
    algos=["State"],
    configs=experiments.configs,
    path_to_data_dir=PATH_TO_DATA_DIR,
    signal_columns=SIGNAL_COLUMNS,
)
config = _iter._get_config_for_size("State", SIZE, dataset=DATASET)
config


[2026-04-17 15:52:00] Max steps for State with 100 cells: 15000; early stopping patience=5


{'max_steps': 15000, 'early_stopping_patience': 5}

In [3]:
# State runs as a subprocess that writes directly to this cell's stdout —
# Lightning's per-step training log streams live (no capturing).
experiments.single_job(
    dataset=DATASET,
    size=SIZE,
    quality=QUALITY,
    algo="State",
    max_epochs=config.get("max_epochs"),
    early_stopping_patience=config["early_stopping_patience"],
    max_steps=config.get("max_steps"),
    device=DEVICE,
    retrain=True,
    reembed=True,
    recompute_mutual_information=True,
)


[2026-04-17 15:52:00] Running State for merfish with 100 cells and 1.0 quality on device 0 (using max_epochs=None and early_stopping_patience=5)


[2026-04-17 15:52:01] Using GPU 0 (visible as cuda:0)
[2026-04-17 15:52:01] Running State for merfish with 100 cells and 1.0 quality on device 0 (using max_epochs=None and early_stopping_patience=5)
[2026-04-17 15:52:01]   Data: 100 cells, ~1 batches/epoch, val every 500 steps
[2026-04-17 15:52:01]   Step budget: 15000 steps → 15000 epochs (with early stopping)
[2026-04-17 15:52:01]   Running: /home/igor/miniconda3/envs/state/bin/python -m state emb fit --conf /home/igor/noise_scaling/data/merfish/100/1.0/preprocessed/state_data/state_config.yaml embeddings.current=scaling_merfish_100_1_0 dataset.current=scaling_merfish_100_1_0 dataset.num_cells=100 dataset.num_train_workers=4 dataset.num_val_workers=2 dataset.pad_length=2048 dataset.P=512 dataset.N=512 dataset.S=512 model.batch_size=64 model.emsize=256 model.d_hid=512 model.nhead=4 model.nlayers=3 model.output_dim=256 model.dataset_correction=false model.dropout=0.1 optimizer.max_lr=0.0001 optimizer.gradient_accumulation_steps=1 optim

Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
You are using a CUDA device ('NVIDIA A100-SXM4-40GB') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loading `train_dataloader` to estimate number of stepping batches.


Starting training with Embedding scaling_merfish_100_1_0 and dataset scaling_merfish_100_1_0
  Step-based early stopping: monitor=validation/val_loss, every_n_steps=500, patience=5
******** Initialized fresh exp_state_scaling_merfish_100_1_0_layers_3_dmodel_256_samples_2048_max_lr_0.0001_op_dim_256...
┏━━━┳━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name                ┃ Type                    ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ encoder             │ Sequential              │  656 K │ train │     0 │
│ 1 │ transformer_encoder │ FlashTransformerEncoder │  1.6 M │ train │     0 │
│ 2 │ decoder             │ Sequential              │  329 K │ train │     0 │
│ 3 │ binary_decoder      │ Sequential              │  2.1 M │ train │     0 │
│ 4 │ bin_encoder         │ Embedding               │  2.6 K │ train │     0 │
│ 5 │ count_encoder       │ Sequential              │  6.2 K │ tr

Epoch 0/14999 ━━━━━━━━━━━━━━━━━━━━━━ 0/2 0:00:00 • -:--:-- 0.00it/s v_num: 0.000

...

Epoch 95/14999 ━━━━━━━━━━━━━━━━━━━━━ 0/2 0:00:00 • -:--:-- 0.00it/s v_num: 0.000

Epoch 95/14999 ━━━━━━━━━━━━━━━━━━━━ 2/2 0:00:00 • 0:00:00 11.26it/s v_num: 0.000


Traceback (most recent call last):
  File "/home/igor/miniconda3/envs/state/lib/python3.11/site-packages/lightning/pytorch/trainer/call.py", line 49, in _call_and_handle_interrupt
    return trainer_fn(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/igor/miniconda3/envs/state/lib/python3.11/site-packages/lightning/pytorch/trainer/trainer.py", line 630, in _fit_impl
    self._run(model, ckpt_path=ckpt_path, weights_only=weights_only)
  File "/home/igor/miniconda3/envs/state/lib/python3.11/site-packages/lightning/pytorch/trainer/trainer.py", line 1079, in _run
    results = self._run_stage()
              ^^^^^^^^^^^^^^^^^
  File "/home/igor/miniconda3/envs/state/lib/python3.11/site-packages/lightning/pytorch/trainer/trainer.py", line 1123, in _run_stage
    self.fit_loop.run()
  File "/home/igor/miniconda3/envs/state/lib/python3.11/site-packages/lightning/pytorch/loops/fit_loop.py", line 217, in run
    self.advance()
  File "/home/igor/miniconda3/envs/state/lib/py

[2026-04-17 15:53:12]   Loss curves saved to /home/igor/noise_scaling/data/merfish/100/1.0/results/State/model/loss/metrics.csv (train_pts=48, val_pts=0, last_step=47)
[2026-04-17 15:53:12]   Final train loss: 30.060896 → /home/igor/noise_scaling/data/merfish/100/1.0/results/State/model/train_loss.txt


CalledProcessError: Command '['/home/igor/miniconda3/envs/state/bin/python', '-m', 'state', 'emb', 'fit', '--conf', '/home/igor/noise_scaling/data/merfish/100/1.0/preprocessed/state_data/state_config.yaml', 'embeddings.current=scaling_merfish_100_1_0', 'dataset.current=scaling_merfish_100_1_0', 'dataset.num_cells=100', 'dataset.num_train_workers=4', 'dataset.num_val_workers=2', 'dataset.pad_length=2048', 'dataset.P=512', 'dataset.N=512', 'dataset.S=512', 'model.batch_size=64', 'model.emsize=256', 'model.d_hid=512', 'model.nhead=4', 'model.nlayers=3', 'model.output_dim=256', 'model.dataset_correction=false', 'model.dropout=0.1', 'optimizer.max_lr=0.0001', 'optimizer.gradient_accumulation_steps=1', 'optimizer.weight_decay=0.01', 'experiment.name=state_scaling_merfish_100_1_0', 'experiment.num_epochs=15000', 'experiment.num_gpus_per_node=1', 'experiment.num_nodes=1', 'experiment.port=45485', 'experiment.val_check_interval=500', '+experiment.log_every_n_steps=10', 'experiment.limit_val_bat

In [4]:
# Quick peek at the scalar loss files written next to the model checkpoint.
from pathlib import Path

model_dir = Path(PATH_TO_DATA_DIR) / DATASET / str(SIZE) / str(QUALITY) / "results" / "State" / "model"
for name in ("train_loss.txt", "val_loss.txt", "test_loss.txt"):
    p = model_dir / name
    print(f"{name}: {p.read_text().strip() if p.exists() else '<missing>'}")


[2026-04-17 15:53:12] train_loss.txt: 30.060896
[2026-04-17 15:53:12] val_loss.txt: 33.896812
[2026-04-17 15:53:12] test_loss.txt: 33.896812
